<a href="https://colab.research.google.com/github/Legend-Raj/Final_Year_Project_Progress/blob/main/scrapper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
Extract the detailed job information from the LinkedIn posting at "https://www.linkedin.com/jobs/view/4376334884" by following these steps:

1.  **Consolidate Environment and Helpers**: Create a single setup cell that includes all essential imports (`selenium`, `BeautifulSoup`, `pandas`, `spacy`, `re`), the `get_resilient_driver` configuration, the `expanded_selectors_map`, and the corrected NLP helper functions for extracting technical skills (using `re.escape` for terms like C++) and salary ranges.
2.  **Consolidate Extraction Logic**: Implement a unified extraction function that uses `WebDriverWait` to ensure the job detail pane is fully rendered. Apply the `expanded_selectors_map` to capture the `job_title`, `company`, `location`, `job_desc`, `employment_type`, and `industry`.
3.  **Process and Export**: Execute the extraction for the specific URL, convert the resulting data into a structured Pandas DataFrame, standardize any missing values or placeholders to 'Not Available', and generate a consolidated 'text' column merging title, company, location, and description.
4.  **Final Task**: Export the validated single-row dataset to a CSV file named "linkedin_jobs.csv" and display the final DataFrame row to confirm successful completion.

## Consolidate Clean Environment and Helpers

### Subtask:
Create a single, self-contained setup cell for all required libraries, configurations, and utility functions.


**Reasoning**:
I will consolidate all required imports, utility functions, selector maps, and NLP extraction logic into a single, self-contained setup block to ensure environment stability and cleaner code for the subsequent extraction steps.



In [4]:
!pip install selenium
!apt-get update
!apt-get install -y google-chrome-stable
print('Dependencies installed successfully.')
import pandas as pd
import re
import random
import time
import json
import spacy
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Initialize SpaCy
try:
    nlp = spacy.load('en_core_web_sm')
except:
    import os
    os.system('python -m spacy download en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

def get_resilient_driver():
    user_agents = [
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36',
        'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
    ]
    chrome_options = Options()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument(f'user-agent={random.choice(user_agents)}')
    driver = webdriver.Chrome(options=chrome_options)
    return driver

expanded_selectors_map = {
    'title': ['h1', '.top-card-layout__title', '.jobs-unified-top-card__job-title'],
    'company': ['.topcard__org-name-link', '.jobs-unified-top-card__company-name', '.base-search-card__subtitle'],
    'location': ['.topcard__flavor--bullet', '.jobs-unified-top-card__bullet', 'span.topcard__flavor'],
    'description': ['.show-more-less-html__markup', '.jobs-description-content__text', '#job-details'],
    'employment_type': ['.description__job-criteria-item'],
    'industry': ['.description__job-criteria-item']
}

def get_text_from_selectors(soup, selectors):
    for selector in selectors:
        element = soup.select_one(selector)
        if element: return element.get_text(separator=' ', strip=True)
    return "Not Available"

def extract_skills_robust(text):
    if not text or text == 'Not Available': return 'Not Available'
    skill_keywords = ['Python', 'SQL', 'R', 'Machine Learning', 'Deep Learning', 'Tableau', 'Power BI', 'AWS', 'Azure', 'Spark', 'Hadoop', 'Java', 'C++', 'TensorFlow', 'PyTorch', 'Excel']
    found_skills = [skill for skill in skill_keywords if re.search(rf'\b{re.escape(skill)}\b', text, re.IGNORECASE)]
    return ", ".join(list(set(found_skills))) if found_skills else 'Not Available'

def extract_salary_robust(text):
    if not text or text == 'Not Available': return 'Not Available'
    salary_pattern = r'(\$[0-9]{1,3}(?:,[0-9]{3})*(?:\s?[kK])?(?:\s?- ?\$?[0-9]{1,3}(?:,[0-9]{3})*(?:\s?[kK])?)?)'
    matches = re.findall(salary_pattern, text)
    return " / ".join(matches) if matches else 'Not Available'

print("Environment and Selectors restored to original state.")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,778 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package google-chrome-stable
D

## Consolidate Extraction Logic

### Subtask:
Create a clean, unified extraction function cell that incorporates the explicit 'Wait for Element' logic and the multi-selector strategy for reliable data capture.


**Reasoning**:
I will define the `extract_single_job` function which consolidates the navigation, waiting, BeautifulSoup parsing, and NLP enrichment logic into a single reusable component for extracting details from a specific job URL.



In [5]:
def extract_linkedin_job(driver, url):
    """Navigates to a specific LinkedIn job URL and extracts detailed, enriched data."""
    print(f"Navigating to: {url}")
    driver.get(url)

    wait = WebDriverWait(driver, 10)
    try:
        # 3. Wait until the job title or description is present in the DOM
        wait.until(EC.any_of(
            EC.presence_of_element_located((By.CSS_SELECTOR, expanded_selectors_map['title'][0])),
            EC.presence_of_element_located((By.CSS_SELECTOR, expanded_selectors_map['description'][0]))
        ))
    except TimeoutException:
        print("Warning: Timeout reached while waiting for page content to render.")

    # 4. Parse the rendered page source with BeautifulSoup
    soup = BeautifulSoup(driver.page_source, 'html.parser')

    # 5. Extract fields using get_text_from_selectors and expanded_selectors_map
    job_data = {
        'job_title': get_text_from_selectors(soup, expanded_selectors_map['title']),
        'company': get_text_from_selectors(soup, expanded_selectors_map['company']),
        'location': get_text_from_selectors(soup, expanded_selectors_map['location']),
        'job_desc': get_text_from_selectors(soup, expanded_selectors_map['description']),
        'employment_type': get_text_from_selectors(soup, expanded_selectors_map['employment_type']),
        'industry': get_text_from_selectors(soup, expanded_selectors_map['industry'])
    }

    # 6. Apply NLP and Regex helpers for skills and salary enrichment
    job_data['skills_desc'] = extract_skills_robust(job_data['job_desc'])
    job_data['salary_range'] = extract_salary_robust(job_data['job_desc'])

    return job_data

print("Unified extraction function 'extract_linkedin_job' defined.")

Unified extraction function 'extract_linkedin_job' defined.


## Consolidate Transformation and Export

### Subtask:
Execute the extraction for the target URL, convert the data into a Pandas DataFrame, clean placeholders, and export to CSV.


**Reasoning**:
I will execute the extraction for the target LinkedIn URL, process the returned dictionary into a standardized Pandas DataFrame, create the consolidated 'text' column, and export the result to 'linkedin_jobs.csv'.



In [6]:
import os

def get_target_url_from_user():
    """Prompts the user to enter a LinkedIn job URL and returns it."""
    return input("Please enter the LinkedIn job URL to extract: ")

def clear_duplicates_from_dataframe(df):
    """Removes duplicate entries from the DataFrame based on key job fields."""
    initial_rows = len(df)
    subset_cols = ['job_title', 'company', 'location', 'job_desc']
    df_cleaned = df.drop_duplicates(subset=subset_cols, keep='first')
    removed_rows = initial_rows - len(df_cleaned)
    if removed_rows > 0:
        print(f"Removed {removed_rows} duplicate job entries.")
    else:
        print("No duplicate job entries found.")
    return df_cleaned

# 1. Get the target URL from the user
target_url = get_target_url_from_user()

# Initialize the WebDriver
driver = get_resilient_driver()

try:
    # Call the extraction function for the target URL
    job_details = extract_linkedin_job(driver, target_url)

    # Convert the dictionary to a list and then a Pandas DataFrame
    new_job_df = pd.DataFrame([job_details])

    # Replace placeholder values with 'Not Available' for the new job
    placeholders = ['N/A', 'Unknown', '', None]
    new_job_df = new_job_df.replace(placeholders, 'Not Available')

    # Create the consolidated 'text' column for the new job
    new_job_df['text'] = (
        new_job_df['job_title'].astype(str) + "\n" +
        new_job_df['company'].astype(str) + "\n" +
        new_job_df['location'].astype(str) + "\n" +
        new_job_df['job_desc'].astype(str)
    )

    # Load existing data if the CSV file exists, otherwise start with the new job
    if os.path.exists('linkedin_jobs.csv'):
        existing_df = pd.read_csv('linkedin_jobs.csv')

        # Align columns before concatenation
        all_cols = list(set(existing_df.columns) | set(new_job_df.columns))
        for col in all_cols:
            if col not in existing_df.columns:
                existing_df[col] = 'Not Available'
            if col not in new_job_df.columns:
                new_job_df[col] = 'Not Available'

        # Ensure the order of columns is consistent before concatenation
        new_job_df = new_job_df[existing_df.columns]

        df_job_final = pd.concat([existing_df, new_job_df], ignore_index=True)
    else:
        df_job_final = new_job_df

    # Clear duplicate entries
    df_job_final = clear_duplicates_from_dataframe(df_job_final)

    # Export to CSV
    df_job_final.to_csv('linkedin_jobs.csv', index=False)

    # Display the DataFrame
    print("Data extraction and transformation complete. Updated DataFrame:")
    display(df_job_final)

finally:
    # Close the WebDriver session
    driver.quit()
    print("WebDriver session closed.")

Please enter the LinkedIn job URL to extract: https://www.linkedin.com/jobs/view/4384232495
Navigating to: https://www.linkedin.com/jobs/view/4384232495
No duplicate job entries found.
Data extraction and transformation complete. Updated DataFrame:


,job_title,company,location,job_desc,employment_type,industry,skills_desc,salary_range,text
0,AI Automator & Researcher,NexGen Media,"Pune Division, Maharashtra, India",AI Automation & Research Intern Company: NexGe...,Seniority level Internship,Seniority level Internship,Python,Not Available,AI Automator & Researcher\nNexGen Media\nPune ...


WebDriver session closed.


Naukri.com

In [22]:
!pip install playwright playwright-stealth nest_asyncio
!playwright install
!apt-get update
!apt-get install -y libxcomposite1 libgtk-3-0 libatk1.0-0 libgstreamer1.0-0 libgstreamer-plugins-base1.0-0 libgstreamer-plugins-good1.0-0 libgstreamer-plugins-bad1.0-0 libavif-dev libharfbuzz-icu0 libmanette-0.2-0 libenchant-2-2 libhyphen0 libsecret-1-0 libwoff2-1.0.2

import pandas as pd
import re
import random
import spacy
from bs4 import BeautifulSoup
import os
from IPython.display import display
import asyncio
import nest_asyncio
from playwright.async_api import async_playwright
from playwright_stealth import stealth

nest_asyncio.apply()

try:
    nlp = spacy.load('en_core_web_sm')
except:
    import os
    os.system('python -m spacy download en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

async def get_resilient_driver():
    user_agents = [
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36',
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/118.0.0.0 Safari/537.36'
    ]
    pw = await async_playwright().start()
    browser = await pw.chromium.launch(headless=True)
    context = await browser.new_context(user_agent=random.choice(user_agents), viewport={'width': 1280, 'height': 800})
    page = await context.new_page()
    await stealth(page)
    return page, browser, pw

async def get_text_from_selectors(soup, selectors):
    for selector in selectors:
        element = soup.select_one(selector)
        if element and element.get_text(strip=True):
            return element.get_text(separator=' ', strip=True)
    return "N/A"

def extract_skills_robust(text):
    if not text or text == 'N/A': return 'Not Available'
    skill_keywords = ['Python', 'SQL', 'R', 'Machine Learning', 'Deep Learning', 'Tableau', 'Power BI', 'AWS', 'Azure', 'Spark', 'Hadoop', 'Java', 'C++', 'TensorFlow', 'PyTorch', 'Excel']
    found_skills = [skill for skill in skill_keywords if re.search(rf'\b{re.escape(skill)}\b', text, re.IGNORECASE)]
    return ", ".join(list(set(found_skills))) if found_skills else 'Not Available'

def extract_salary_robust(text):
    if not text or text == 'N/A': return 'Not Available'
    salary_pattern = r'(\$[0-9]{1,3}(?:,[0-9]{3})*(?:\s?[kK])?(?:\s?- ?\$?[0-9]{1,3}(?:,[0-9]{3})*(?:\s?[kK])?)?)'
    matches = re.findall(salary_pattern, text)
    return " / ".join(matches) if matches else 'Not Available'

naukri_selectors_map = {
    'title': ['h1.jd-header-title', 'h1.job-description-title'],
    'company': ['a.pad-rt-8', 'a.chip.chip-primary.job-description-chip'],
    'location': ['div.location', 'span.location'],
    'description': ['div.job-desc', 'div.dang-inner-html']
}

async def main():
    target_url = input("Please enter the Naukri job URL: ")
    page, browser, pw = await get_resilient_driver()
    try:
        await page.goto(target_url, wait_until='domcontentloaded')
        await page.wait_for_timeout(5000)
        soup = BeautifulSoup(await page.content(), 'html.parser')
        job_data = {
            'job_title': await get_text_from_selectors(soup, naukri_selectors_map['title']),
            'company': await get_text_from_selectors(soup, naukri_selectors_map['company']),
            'location': await get_text_from_selectors(soup, naukri_selectors_map['location']),
            'job_desc': await get_text_from_selectors(soup, naukri_selectors_map['description'])
        }
        job_data['skills_desc'] = extract_skills_robust(job_data['job_desc'])
        job_data['salary_range'] = extract_salary_robust(job_data['job_desc'])
        df = pd.DataFrame([job_data])
        display(df)
        df.to_csv('naukri_jobs.csv', index=False)
    finally:
        await browser.close()
        await pw.stop()

await main()

Playwright Host validation warning: 
╔══════════════════════════════════════════════════════╗
║ Host system is missing dependencies to run browsers. ║
║ Missing libraries:                                   ║
║     libgstgl-1.0.so.0                                ║
║     libgstcodecparsers-1.0.so.0                      ║
║     libwoff2dec.so.1.0.2                             ║
╚══════════════════════════════════════════════════════╝
    at validateDependenciesLinux (/usr/local/lib/python3.12/dist-packages/playwright/driver/package/lib/server/registry/dependencies.js:269:9)
    at process.processTicksAndRejections (node:internal/process/task_queues:103:5)
    at async Registry._validateHostRequirements (/usr/local/lib/python3.12/dist-packages/playwright/driver/package/lib/server/registry/index.js:991:14)
    at async Registry._validateHostRequirementsForExecutableIfNeeded (/usr/local/lib/python3.12/dist-packages/playwright/driver/package/lib/server/registry/index.js:1113:7)
    at async 

TypeError: 'module' object is not callable

In [23]:
!pip install requests-html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.9/84.9 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.9/82.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 5.1 MB/s eta 0:00:00
  Created wheel for websockets: filename=websockets-10.4-cp312-cp312-linux_x86_64.whl size=107328 sha256=dbe7728b0a89c47e72caff9928754e3bb84c63806d382ed3a1d5739a1fc2b0f3
  Stored in directory: /root/.cache/pip/wheels/80/cf/6d/5d7e4c920cb41925a178b2d2621889c520d648bab487b1d7fd
Successfully built websockets
  Attempting uninstall: websockets
    Found existing installation: websockets 15.0.1
    Uninstalling websockets-15.0.1:
      Successfully uninstalled websockets-15.0.1
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    

In [1]:
!pip install lxml_html_clean

In [4]:
from requests_html import AsyncHTMLSession
import pandas as pd
import re
from bs4 import BeautifulSoup
from IPython.display import display
import nest_asyncio

# Allow nested event loops for Colab
nest_asyncio.apply()

async def extract_naukri_async(url):
    session = AsyncHTMLSession()
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
    }

    print(f"Fetching URL: {url}")
    response = await session.get(url, headers=headers)

    print("Rendering JavaScript (this may take a few seconds)...")
    await response.html.arender(timeout=20, sleep=5)

    soup = BeautifulSoup(response.html.html, 'html.parser')

    # Extended Selectors for Naukri
    title = soup.select_one('h1.jd-header-title, h1.job-description-title')
    company = soup.select_one('a.pad-rt-8, a.chip.chip-primary.job-description-chip, .jd-header-comp-name')
    location = soup.select_one('div.location, span.location, .loc .label')
    desc = soup.select_one('div.job-desc, div.dang-inner-html, .job-desc-text')

    # Attempt to find job insights (Employment Type/Industry)
    details_section = soup.select('.details .other-details .details-item')
    emp_type = "Not Available"
    industry = "Not Available"

    for item in details_section:
        label = item.select_one('label')
        if label:
            if "Employment Type" in label.get_text():
                emp_type = item.select_one('span').get_text(strip=True) if item.select_one('span') else "Not Available"
            if "Industry" in label.get_text() or "Role Category" in label.get_text():
                industry = item.select_one('span').get_text(strip=True) if item.select_one('span') else "Not Available"

    job_data = {
        'job_title': title.get_text(strip=True) if title else 'Not Available',
        'company': company.get_text(strip=True) if company else 'Not Available',
        'location': location.get_text(strip=True) if location else 'Not Available',
        'job_desc': desc.get_text(separator=' ', strip=True) if desc else 'Not Available',
        'employment_type': emp_type,
        'industry': industry
    }

    # Enrichment logic matching LinkedIn version
    skill_keywords = ['Python', 'SQL', 'Machine Learning', 'Java', 'AWS', 'Excel', 'Sales', 'C++']
    found_skills = [s for s in skill_keywords if re.search(rf'\b{re.escape(s)}\b', job_data['job_desc'], re.I)]
    job_data['skills_desc'] = ", ".join(found_skills) if found_skills else 'Not Available'

    salary_pattern = r'(\d{1,2}(?:\.\d+)?\s?-\s?\d{1,2}(?:\.\d+)?\s?P\.A\.)'
    salary_match = re.search(salary_pattern, response.html.html)
    job_data['salary_range'] = salary_match.group(1) if salary_match else 'Not Available'

    await session.close()
    return job_data

# Execution
target_url = input("Please enter the Naukri job URL: ")
try:
    data = await extract_naukri_async(target_url)
    df = pd.DataFrame([data])

    # Create consolidated 'text' column for RAG/Similarity tasks
    df['text'] = (
        df['job_title'].astype(str) + "\n" +
        df['company'].astype(str) + "\n" +
        df['location'].astype(str) + "\n" +
        df['job_desc'].astype(str)
    )

    display(df)
    df.to_csv('naukri_jobs_alt.csv', index=False)
    print("\nDetailed extraction complete and saved to naukri_jobs_alt.csv")
except Exception as e:
    print(f"\nAn error occurred: {e}")

Please enter the Naukri job URL: https://www.naukri.com/job-listings-sales-specialist-apple-part-time-apple-mumbai-new-delhi-bengaluru-0-to-4-years-240326022281
Fetching URL: https://www.naukri.com/job-listings-sales-specialist-apple-part-time-apple-mumbai-new-delhi-bengaluru-0-to-4-years-240326022281
Rendering JavaScript (this may take a few seconds)...


,job_title,company,location,job_desc,employment_type,industry,skills_desc,salary_range,text
0,Not Available,Not Available,Not Available,Not Available,Not Available,Not Available,Not Available,Not Available,Not Available\nNot Available\nNot Available\nN...



Detailed extraction complete and saved to naukri_jobs_alt.csv


In [6]:
import requests
import pandas as pd
import time
from IPython.display import display

def search_adzuna_jobs(app_id, app_key, query, location='us', results_per_page=10):
    """Searches for jobs using the Adzuna API and returns a structured DataFrame."""
    base_url = f"https://api.adzuna.com/v1/api/jobs/{location}/search/1"

    params = {
        'app_id': app_id,
        'app_key': app_key,
        'results_per_page': results_per_page,
        'what': query,
        'content-type': 'application/json'
    }

    print(f"Searching Adzuna for: '{query}' in {location}...")
    response = requests.get(base_url, params=params)

    if response.status_code != 200:
        print(f"Error: {response.status_code} - {response.text}")
        return None

    data = response.json()
    jobs = data.get('results', [])

    processed_jobs = []
    for job in jobs:
        # Standardizing to our existing schema
        processed_jobs.append({
            'job_title': job.get('title', 'Not Available'),
            'company': job.get('company', {}).get('display_name', 'Not Available'),
            'location': job.get('location', {}).get('display_name', 'Not Available'),
            'job_desc': job.get('description', 'Not Available'),
            'employment_type': job.get('contract_type', 'Not Available'),
            'industry': job.get('category', {}).get('label', 'Not Available'),
            'salary_range': f"{job.get('salary_min', '')} - {job.get('salary_max', '')}" if job.get('salary_min') else 'Not Available',
            'url': job.get('redirect_url')
        })

    df = pd.DataFrame(processed_jobs)

    # Add the 'text' column for RAG/Similarity compatibility
    if not df.empty:
        df['text'] = (
            df['job_title'].astype(str) + "\n" +
            df['company'].astype(str) + "\n" +
            df['location'].astype(str) + "\n" +
            df['job_desc'].astype(str)
        )

    return df

In [7]:
# Replace these with your actual Adzuna API credentials
APP_ID = 'YOUR_ADZUNA_APP_ID'
APP_KEY = 'YOUR_ADZUNA_APP_KEY'

# Example Search
search_query = input("Enter job title or keyword to search: ")
job_results_df = search_adzuna_jobs(APP_ID, APP_KEY, search_query)

if job_results_df is not null and not job_results_df.empty:
    display(job_results_df.head())
    job_results_df.to_csv('adzuna_jobs.csv', index=False)
    print(f"\nSaved {len(job_results_df)} results to adzuna_jobs.csv")
else:
    print("No results found or API error occurred.")

KeyboardInterrupt: Interrupted by user

In [5]:
from requests_html import AsyncHTMLSession
import nest_asyncio
nest_asyncio.apply()

async def debug_naukri(url):
    session = AsyncHTMLSession()
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'}
    response = await session.get(url, headers=headers)
    await response.html.arender(timeout=20, sleep=5)

    # Print the first 2000 characters to see the tag structure
    print("HTML Snippet (Body Start):")
    print(response.html.html[:2000])

    await session.close()

# Run debug for the specific URL
await debug_naukri('https://www.naukri.com/job-listings-sales-specialist-apple-part-time-apple-mumbai-new-delhi-bengaluru-0-to-4-years-240326022281')

HTML Snippet (Body Start):
<html><head>
<title>Access Denied</title>
</head><body>
<h1>Access Denied</h1>
 
You don't have permission to access "http://www.naukri.com/job-listings-sales-specialist-apple-part-time-apple-mumbai-new-delhi-bengaluru-0-to-4-years-240326022281" on this server.<p>
Reference #18.b200117.1775734861.622c5106
</p><p>https://errors.edgesuite.net/18.b200117.1775734861.622c5106</p>


</body></html>
